# Cassette corners: one matched occlusion-training experiment

Choose a **CUDA GPU**, then **Runtime → Run all**. Approve the Drive mount to preserve completed work. Return **`baomai-occlusion-seed46-results.zip`** when downloaded.

Both runs use the exact PR35 annotation composition, seed 46, 30 epochs, input 1280, batch 4, rectangular loading, FP32 and fresh original epoch173 initialization. Both retain the original training/validation split. The control plans the same corruptions but applies none; the candidate applies only the declared training-pixel corruption. Source coordinates and labels remain unchanged, including missing slots. The experiment tests robustness to corrupted input, not independently observed hidden coordinates.

Completed compatible runs are restored and skipped. Partial or incompatible runs remain preserved and cannot count as complete. A retry needs a fresh `RUN_BASE`. Matching source, recipe and GPU/runtime are required for the pair. The resource trial is not an eligible model and never initializes a full run.


## 1. Prepare the pinned workspace and Drive backup

This notebook refuses unresolved pins. Keep the supplied values unchanged.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REF = "PENDING_IMPLEMENTATION_COMMIT"
PROTOCOL_PATH = "experiments/occlusion-training-2026-11-13.json"
PROTOCOL_SHA256 = "c70563034d3ac969ff7837a594d53bc6316fda3897e1557b5d7828be52e94a62"
RUN_BASE = Path("/content/baomai-occlusion-seed46")
ROLES = ("control", "candidate")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF), "Use the final handoff notebook with its full source commit pinned."
assert re.fullmatch(r"[0-9a-f]{64}", PROTOCOL_SHA256), "Use the final handoff notebook with its protocol hash pinned."
assert sys.version_info >= (3, 11), "Python3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

workspace_identity = {"source_revision": SOURCE_REF, "protocol_sha256": PROTOCOL_SHA256}
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    assert RUN_BASE.is_dir() and not RUN_BASE.is_symlink(), "Choose a fresh nonsymlink workspace."
    marker = RUN_BASE / "workspace-identity.json"
    assert marker.is_file() and json.loads(marker.read_text()) == workspace_identity, "Existing workspace belongs to different pins; preserve it and choose a new RUN_BASE."
else:
    RUN_BASE.mkdir(parents=True)
    (RUN_BASE / "workspace-identity.json").write_text(json.dumps(workspace_identity, indent=2) + "\n")
REPO = RUN_BASE / "dl"
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Drive is unavailable here; preserve completed bundles and the ZIP from the local workspace.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-occlusion-seed46-20261113")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def safe_member(name):
    p = Path(name)
    assert isinstance(name, str) and name and not p.is_absolute() and ".." not in p.parts
    assert "\\" not in name and ":" not in name and p.as_posix() == name
    return p

def artifact_inventory(path):
    path = Path(path)
    if path.is_file():
        names = [path.name]
        base = path.parent
    else:
        base = path
        receipt = json.loads((path / "training-receipt.json").read_text())
        names = ["training-receipt.json", "staging-manifest.json", receipt["log"]["path"]]
        if (path / "checkpoint-manifest.json").is_file():
            manifest = json.loads((path / "checkpoint-manifest.json").read_text())
            names += ["checkpoint-manifest.json", manifest["checkpoint"]["path"], receipt["results"]["path"]]
    assert len(names) == len(set(names))
    return [{"path": name, "sha256": sha256(base / safe_member(name)), "size_bytes": (base / name).stat().st_size}
            for name in sorted(names)]

def validate_copy(path, marker):
    base = path if marker["directory"] else path.parent
    for entry in marker["files"]:
        target = base / safe_member(entry["path"])
        assert target.is_file() and not target.is_symlink(), f"Missing backup artifact: {target}"
        assert target.stat().st_size == entry["size_bytes"] and sha256(target) == entry["sha256"], f"Backup hash mismatch: {target}"

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    relative = path.relative_to(RUN_BASE).as_posix()
    if relative == "resource-trial":
        relative = "resource-trials/" + sha256(path / "training-receipt.json")[:20]
    destination = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(destination) + ".complete")
    marker = {**workspace_identity, "relative_path": relative, "directory": path.is_dir(), "files": artifact_inventory(path)}
    if marker_path.is_file():
        assert json.loads(marker_path.read_text()) == marker, f"Drive already has different completed evidence: {destination}"
        validate_copy(destination, marker)
        print("Verified existing Drive backup:", destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Only an incomplete copy in this notebook's exact destination is replaced.
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    if path.is_dir():
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path / entry["path"], target)
    else:
        shutil.copyfile(path, destination)
    validate_copy(destination, marker)
    marker_path.write_text(json.dumps(marker, indent=2) + "\n")
    print("Saved verified completed artifacts:", destination, flush=True)

def restore_from_drive(relative):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(source) + ".complete")
    if not marker_path.is_file():
        return None
    marker = json.loads(marker_path.read_text())
    assert all(marker.get(k) == v for k, v in workspace_identity.items()), "Drive completion belongs to different source/protocol pins."
    assert marker["relative_path"] == relative
    validate_copy(source, marker)
    destination = RUN_BASE / safe_member(relative)
    if destination.exists() or destination.is_symlink():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if marker["directory"]:
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / entry["path"], target)
    else:
        assert len(marker["files"]) == 1 and marker["files"][0]["path"] == source.name
        shutil.copyfile(source, destination)
    validate_copy(destination, marker)
    print("Restored completed evidence:", destination, flush=True)
    return destination

print("Workspace:", RUN_BASE)


## 2. Restore and verify the original inputs

Download 137 training PNGs, 20 validation PNGs and original epoch173 weights through Git LFS. Verify the exact PR35 label composition and the fixed protocol before training. Adjudicated development references stay separate from the original in-training validation labels.


In [ ]:
if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
if not REPO.exists():
    REPO.mkdir()
    run(["git", "init"], cwd=REPO)
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
    run(["git", "fetch", "--depth=1", "origin", SOURCE_REF], cwd=REPO, env=git_env)
    run(["git", "checkout", "--detach", SOURCE_REF], cwd=REPO, env=git_env)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_REF
assert sha256(REPO / PROTOCOL_PATH) == PROTOCOL_SHA256
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip(), "Keep the pinned source clean."
run(["git", "lfs", "install", "--local"], cwd=REPO)
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt", "--exclude="], cwd=REPO)
sys.dont_write_bytecode = True
if str(REPO / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO / "scripts"))
from occlusion_experiment import load_protocol, stage_pr35, source_identity, experiment_identity, verify_run, package_pair
from evaluate_pose import preflight_dataset
from pose_checkpoint import get_recipe, recipe_identity
from pose_input_corruption import validate_state
from tempfile import TemporaryDirectory
CONTEXT = load_protocol(REPO, PROTOCOL_PATH)
assert CONTEXT["sha256"] == PROTOCOL_SHA256
assert CONTEXT["document"]["recipe_id"] == "reviewed-annotations-seed46-v1"
assert CONTEXT["document"]["candidate_manifest"] == "data/candidates/annotations-2026-11-10/manifest.json"
PREFLIGHT = preflight_dataset(REPO, require_weights=True)
assert PREFLIGHT["model_sha256"] == CONTEXT["document"]["initial_checkpoint_sha256"]
with TemporaryDirectory(prefix="input-preflight-", dir=RUN_BASE) as temporary:
    staging = stage_pr35(REPO, Path(temporary) / "staged-data")
    assert staging["variants"]["candidate"]["counts"]["train"]["images"] == 137
    assert staging["variants"]["candidate"]["counts"]["val"]["images"] == 20
    print("Verified PR35 label counts:", staging["variants"]["candidate"]["counts"])
print("All original inputs and exact PR35 supervision verified; both roles use label arm candidate.")


## 3. Install the pinned trainer

Preserve the installed Torch/torchvision pair while installing Ultralytics 8.4.182. Dependency conflicts stop the workflow.


In [ ]:
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")), encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__)"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Limits: 600 seconds for the trial; 21,600 seconds for each complete run.")


## 4. Restore completed runs and run the resource trial if needed

A four-batch candidate trial must show real parameter updates and applied corruption. Host-RAM limits are separate from GPU VRAM. If both full runs are already complete, this step skips GPU probing and the trial.


In [ ]:
NUMERICAL = recipe_identity(CONTEXT["document"]["recipe_id"])

def assert_finished(output, role, *, trial=False):
    assert role in ROLES and (not trial or role == "candidate")
    output = Path(output)
    if not trial:
        bundle = verify_run(REPO, output, role, CONTEXT)
        assert bundle["receipt"]["source"] == source_identity(REPO), "Completed run uses a different source revision."
        return bundle["receipt"]
    receipt = json.loads((output / "training-receipt.json").read_text())
    assert receipt["status"] == "trial-complete" and receipt["purpose"] == "resource-trial"
    assert receipt["arm"] == "candidate" and receipt["experiment"] == experiment_identity(CONTEXT, role)
    assert receipt["source"] == source_identity(REPO)
    assert all(receipt[k] == v for k, v in NUMERICAL.items())
    assert receipt["initial_checkpoint_sha256"] == CONTEXT["document"]["initial_checkpoint_sha256"]
    assert receipt["inputs_reverified_after_worker"] is True
    assert receipt["execution"]["exit_code"] == 0 and receipt["execution"]["stop_reason"] is None
    assert receipt["batches_completed"] == 4 and receipt["images_processed"] == 16
    assert len(receipt["batch_samples"]) == 4
    for sample in receipt["batch_samples"]:
        shape = sample["shape"]
        assert len(shape) == 4 and shape[:2] == [4, 3] and min(shape[2:]) > 0 and max(shape[2:]) == 1280
    assert all(receipt["training_settings"][k] == v for k, v in get_recipe(NUMERICAL["recipe_id"]).items())
    assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
    validate_state(receipt["input_corruption"], CONTEXT["document"]["augmentation"], role)
    counters = receipt["input_corruption"]["counters"]
    assert counters["images_seen"] == 16 and counters["batches_seen"] == 4 and counters["images_applied"] > 0
    log = output / safe_member(receipt["log"]["path"])
    assert log.is_file() and not log.is_symlink() and sha256(log) == receipt["log"]["sha256"]
    assert receipt["runtime"]["device"].startswith("cuda:")
    assert receipt["staging_manifest_sha256"] == sha256(output / "staging-manifest.json")
    assert not (output / "checkpoint-manifest.json").exists()
    return receipt


def assert_current_runtime(receipt, current, label):
    expected = receipt["runtime"]
    assert all(expected[k] == v for k, v in current.items() if k != "cudnn_version"), f"Current runtime differs from {label}; select a matching runtime before full training."
    assert expected["backend"]["cudnn_version"] == current["cudnn_version"], f"Current cuDNN runtime differs from {label}; no full arm started."


def train_one(role, *, trial=False):
    assert role in ROLES
    assert not trial or role == "candidate", "Resource trials use candidate labels only."
    recipe_id = CONTEXT["document"]["recipe_id"]
    folder = "resource-trial" if trial else role
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        try:
            receipt = assert_finished(output, role, trial=trial)
        except Exception as error:
            raise FileExistsError(f"Preserve incomplete or incompatible run {output}. Use a fresh RUN_BASE for retry; completed Drive arms will be restored. {error}") from error
        print("Skipping verified completed run:", folder, flush=True)
        save_to_drive(output)
        return output, receipt
    command = [sys.executable, "-B", "scripts/train_pose_occlusion.py",
               "--trial" if trial else "--train", "--role", role, "--protocol", PROTOCOL_PATH,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, role, recipe_id, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}
    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass
    started, last_notice = time.monotonic(), -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}:{elapsed:.0f}s; status={progress.get('status')}; epochs={len(progress.get('completed_epochs', []))}/30; batches={progress.get('batches_completed', 0)}; updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}:{elapsed:.0f}s; verifying and staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed; preserve its receipt/log and use a fresh workspace for retry.")
    except BaseException:
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = assert_finished(output, role, trial=trial)
    save_to_drive(output)
    return output, receipt


def recover_trial(current):
    local = RUN_BASE / "resource-trial"
    if local.exists() or local.is_symlink():
        return train_one("candidate", trial=True)
    if DRIVE_ROOT is not None:
        markers = sorted((DRIVE_ROOT / "resource-trials").glob("*.complete"),
                         key=lambda path: path.stat().st_mtime, reverse=True)
        for marker in markers:
            relative = marker.relative_to(DRIVE_ROOT).as_posix().removesuffix(".complete")
            try:
                restored = restore_from_drive(relative)
                receipt = assert_finished(restored, "candidate", trial=True)
                assert_current_runtime(receipt, current, "resource trial")
                for completed in COMPLETED.values():
                    assert receipt["runtime"] == completed["runtime"], "Saved trial differs from a completed arm."
            except (AssertionError, ValueError, OSError, KeyError, TypeError) as error:
                print("Skipped incompatible saved trial:", relative, str(error))
                continue
            print("Reused verified completed trial:", restored)
            return restored, receipt
    return train_one("candidate", trial=True)


COMPLETED = {}
for role in ROLES:
    restore_from_drive(role)
    output = RUN_BASE / role
    if output.exists() or output.is_symlink():
        COMPLETED[role] = assert_finished(output, role)
        print("Recovered eligible completed arm:", role)
TRIAL_RUNTIME = None
if len(COMPLETED) == 2:
    assert COMPLETED["control"]["runtime"] == COMPLETED["candidate"]["runtime"], "Completed GPU/runtime identities differ."
    print("Both arms are complete; no resource trial or training will repeat.")
else:
    assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or choose a larger-memory runtime before training."
    run(["nvidia-smi"])
    probe = r"""
import importlib.metadata as m, json, platform, torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'
assert (torch.ones(1, device='cuda:0') * 2).item() == 2
print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,
'ultralytics':m.version('ultralytics'),'torch_cuda':torch.version.cuda,'platform':platform.platform(),
'device':'cuda:0','device_name':torch.cuda.get_device_name(0),'device_capability':list(torch.cuda.get_device_capability(0)),
'packages':{n:m.version(n) for n in ('torchvision','numpy','Pillow','opencv-python','psutil')},
'cudnn_version':torch.backends.cudnn.version()}))
"""
    current = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True, cwd=REPO))
    for role, receipt in COMPLETED.items():
        assert_current_runtime(receipt, current, f"completed {role}")
    TRIAL, trial_receipt = recover_trial(current)
    assert_current_runtime(trial_receipt, current, "resource trial")
    TRIAL_RUNTIME = trial_receipt["runtime"]
    for role, receipt in COMPLETED.items():
        assert receipt["runtime"] == TRIAL_RUNTIME, f"Actual trial runtime differs from completed {role}; no full arm started."
    print("Resource trial passed; it is not a completed model.")


## 5. Train missing runs

Run the no-corruption control, then the corruption candidate. Each full run starts fresh from original epoch173 and returns finalized epoch 30. The actual label arm is `candidate` for both because both use PR35 supervision; the separate experiment role determines pixel corruption.


In [ ]:
CONTROL_RUN, control_receipt = train_one("control")
if TRIAL_RUNTIME is not None:
    assert control_receipt["runtime"] == TRIAL_RUNTIME, "Control runtime differs from the trial; do not start the other arm."
CANDIDATE_RUN, candidate_receipt = train_one("candidate")
assert control_receipt["runtime"] == candidate_receipt["runtime"], "Completed runs have different GPU/runtime identities."
for role, receipt in (("control", control_receipt), ("candidate", candidate_receipt)):
    print(role, "label arm:", receipt["arm"], "corruption counters:", receipt["input_corruption"]["counters"])
    print("Final checkpoint:", receipt["checkpoint"]["sha256"])


## 6. Verify and package the matched pair

The dedicated verifier checks the augmentation role and exposure, exact PR35 supervision, completed checkpoints, identical numerical settings, source and GPU/runtime, and identical corruption-plan traces. An existing ZIP is verified instead of overwritten.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-occlusion-seed46-results.zip"
restore_from_drive(RESULTS_ZIP.name)
result = package_pair(REPO, CONTROL_RUN, CANDIDATE_RUN, RESULTS_ZIP,
                      verify_existing=RESULTS_ZIP.exists(), protocol=PROTOCOL_PATH)
print(json.dumps(result, indent=2))
save_to_drive(RESULTS_ZIP)


## 7. Return the ZIP

Attach the downloaded ZIP in chat or upload it to the agreed results PR. A verified copy is saved in Drive. Keep completed bundles and logs until evaluation is finished.


In [ ]:
assert RESULTS_ZIP.is_file(), "Complete both runs and packaging first."
from google.colab import files
files.download(str(RESULTS_ZIP))


## Evaluation after the return

Evaluate both final checkpoints and retained PR35 under the fixed clean-image and synthetic-occlusion criteria in the protocol. Keep all designated objects and failures in denominators. A robustness gain must preserve the clean visible-corner task, object completeness, empty-scene behavior and review workload. Training completion alone does not establish improvement, and this notebook does not automatically promote a model or start another search.
